In [78]:
import pandas as pd 
import numpy as np 
import seaborn as sns
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [36]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

Exploratory Data Analysis

In [37]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [38]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [39]:
# missing values
train.isnull().sum().sort_values(ascending=False)

Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
Pclass           0
Survived         0
Sex              0
Parch            0
SibSp            0
Fare             0
Ticket           0
dtype: int64

In [40]:
train['Survived'].value_counts()

Survived
0    549
1    342
Name: count, dtype: int64

In [41]:
train.groupby('Pclass')['Survived'].mean()

Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

In [42]:
train.corr(numeric_only=True)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
PassengerId,1.000000,-0.005007,-0.035144,0.036847,-0.057527,-0.001652,0.012658
Survived,-0.005007,1.000000,-0.338481,-0.077221,-0.035322,0.081629,0.257307
Pclass,-0.035144,-0.338481,1.000000,-0.369226,0.083081,0.018443,-0.549500
Age,0.036847,-0.077221,-0.369226,1.000000,-0.308247,-0.189119,0.096067
SibSp,-0.057527,-0.035322,0.083081,-0.308247,1.000000,0.414838,0.159651
Parch,-0.001652,0.081629,0.018443,-0.189119,0.414838,1.000000,0.216225
Fare,0.012658,0.257307,-0.549500,0.096067,0.159651,0.216225,1.000000


In [43]:
pd.crosstab(train['Sex'], train['Survived'], normalize='index')

Survived,0,1
Sex,,
female,0.257962,0.742038
male,0.811092,0.188908


In [10]:
train.groupby(['Sex', 'Pclass'])['Survived'].mean()

Sex     Pclass
female  1         0.968085
        2         0.921053
        3         0.500000
male    1         0.368852
        2         0.157407
        3         0.135447
Name: Survived, dtype: float64

In [116]:
train['AgeGroup'] = pd.cut(train['Age'], bins=[0,12,18,30,50,80])
train.groupby('AgeGroup')['Survived'].mean()

C:\Users\HiDeve\AppData\Local\Temp\ipykernel_12460\1023880137.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train.groupby('AgeGroup')['Survived'].mean()


AgeGroup
(0, 12]     0.579710
(12, 18]    0.428571
(18, 30]    0.355556
(30, 50]    0.423237
(50, 80]    0.343750
Name: Survived, dtype: float64

In [117]:
train['FareGroup'] = pd.qcut(train['Fare'], 4) 
train.groupby('FareGroup')['Survived'].mean()

C:\Users\HiDeve\AppData\Local\Temp\ipykernel_12460\3981327844.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train.groupby('FareGroup')['Survived'].mean()


FareGroup
(-0.001, 7.91]     0.197309
(7.91, 14.454]     0.303571
(14.454, 31.0]     0.454955
(31.0, 512.329]    0.581081
Name: Survived, dtype: float64

In [118]:
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1 
train.groupby('FamilySize')['Survived'].mean()

FamilySize
1     0.303538
2     0.552795
3     0.578431
4     0.724138
5     0.200000
6     0.136364
7     0.333333
8     0.000000
11    0.000000
Name: Survived, dtype: float64

In [119]:
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1 
train.groupby('FamilySize')['Survived'].mean()

FamilySize
1     0.303538
2     0.552795
3     0.578431
4     0.724138
5     0.200000
6     0.136364
7     0.333333
8     0.000000
11    0.000000
Name: Survived, dtype: float64

In [120]:
train['Cabin_missing'] = train['Cabin'].isnull().astype(int) 
train.groupby('Cabin_missing')['Survived'].mean()

Cabin_missing
0    0.666667
1    0.299854
Name: Survived, dtype: float64

Separation X and Y

Preprocessing

In [44]:
# Name column 
train['Title'] = train['Name'].str.extract("([A-Za-z]+)\.", expand=False)
test['Title'] = test['Name'].str.extract("([A-Za-z]+)\.", expand=False)

In [45]:
# Nan age Column 
train['Age'] = train.groupby('Title')['Age'].transform(lambda x : x.fillna(x.median())) 
test['Age'] = test.groupby('Title')['Age'].transform(lambda x : x.fillna(x.median()))

In [46]:
train['Title'] = train['Title'].replace({
    'Mlle' : 'Miss',    
    'Mme' : 'Mrs',    
    'Ms' : 'Miss'
}) 
test['Title'] = test['Title'].replace({
    'Mlle' : 'Miss',    
    'Mme' : 'Mrs',  
    'Ms' : 'Miss'
})

In [47]:
train["Title"] = train["Title"].replace([
    "Lady","Countess","Capt","Col","Don","Dr","Major","Rev","Sir","Jonkheer","Dona"
], "Rare")

test["Title"] = test["Title"].replace([
    "Lady","Countess","Capt","Col","Don","Dr","Major","Rev","Sir","Jonkheer","Dona"
], "Rare")

In [48]:
# Missing Value Embarked column   
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0]) 
test['Embarked'] = test['Embarked'].fillna(test['Embarked'].mode()[0])

Feature interaction

In [49]:
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1  
test['FamilySize'] = train['SibSp'] + test['Parch'] + 1 

In [50]:
train['IsAlone'] = (train['FamilySize'] == 1 ).astype(int) 
test['IsAlone'] = (test['FamilySize'] == 1).astype(int) 

In [51]:
train['Sex_Pclass'] = train['Sex'].astype(str) + "_" + train['Pclass'].astype(str)
test['Sex_Pclass'] = test['Sex'].astype(str) + "_" + test['Pclass'].astype(str)

In [52]:
cat_cols = ['Title', 'Embarked', 'Sex_Pclass']
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False) 

In [53]:
train_cat = ohe.fit_transform(train[cat_cols]) 
test_cat = ohe.transform(test[cat_cols])

In [54]:
train_cat = pd.DataFrame(train_cat, columns=ohe.get_feature_names_out(cat_cols)) 
test_cat = pd.DataFrame(test_cat, columns=ohe.get_feature_names_out(cat_cols))

In [55]:
train = train.drop(cat_cols, axis=1).reset_index(drop=True) 
test = test.drop(cat_cols, axis=1).reset_index(drop=True) 

train = pd.concat([train, train_cat], axis=1) 
test = pd.concat([test, test_cat], axis=1)

In [56]:
train.drop('Name', axis=1, inplace=True) 
test.drop('Name', axis=1, inplace=True)

In [57]:
# Sex column 
train['Sex'] = train['Sex'].map({"male":0, "female":1}) 
test['Sex'] = test['Sex'].map({"male":0, "female":1})

In [58]:
# Drop Ticket column 
train.drop('Ticket', axis=1, inplace=True)
test.drop('Ticket', axis=1, inplace=True)

In [59]:
# Cabin column 
train['Cabin_known'] = train['Cabin'].notnull().astype(int) 
test['Cabin_known'] = test['Cabin'].notnull().astype(int)

In [60]:
# Drop cabin column 
train.drop('Cabin', axis=1, inplace=True) 
test.drop('Cabin', axis=1, inplace=True)

In [62]:
train.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch',
       'Fare', 'FamilySize', 'IsAlone', 'Title_Master', 'Title_Miss',
       'Title_Mr', 'Title_Mrs', 'Title_Rare', 'Embarked_C', 'Embarked_Q',
       'Embarked_S', 'Sex_Pclass_female_1', 'Sex_Pclass_female_2',
       'Sex_Pclass_female_3', 'Sex_Pclass_male_1', 'Sex_Pclass_male_2',
       'Sex_Pclass_male_3', 'Cabin_known'],
      dtype='object')

In [63]:
train.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,FamilySize,IsAlone,...,Embarked_C,Embarked_Q,Embarked_S,Sex_Pclass_female_1,Sex_Pclass_female_2,Sex_Pclass_female_3,Sex_Pclass_male_1,Sex_Pclass_male_2,Sex_Pclass_male_3,Cabin_known
0,1,0,3,0,22.0,1,0,7.2500,2,0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0
1,2,1,1,1,38.0,1,0,71.2833,2,0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1
2,3,1,3,1,26.0,0,0,7.9250,1,1,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0
3,4,1,1,1,35.0,1,0,53.1000,2,0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1
4,5,0,3,0,35.0,0,0,8.0500,1,1,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0


In [66]:
X = train.drop('Survived', axis=1) 
y = train['Survived']

In [73]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2)

In [84]:
model = RandomForestClassifier(n_estimators=100, random_state=42) 
model.fit(X_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [85]:
y_pred = model.predict(X_test)

In [86]:
pred = model.predict(test)

In [87]:
print(accuracy_score(y_test,y_pred))

0.8324022346368715


In [88]:
submission  = pd.DataFrame({
    'PassengerId' : test['PassengerId'],   
    'Survived' : pred
})

In [89]:
submission.to_csv('submission.csv', index=False)

In [90]:
submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
